# Build the full-history point-in-time 13F matrices

This notebook materializes one sparse manager–security pair table for every decision-complete quarter. Each historical state uses only filings available by that quarter's standardized information cutoff. Generated matrices, manifests, and the executed notebook remain under ignored local `data/` paths because they contain raw manager CIKs and reported CUSIPs.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "matrix_13f.py").is_file():
    raise RuntimeError("Run this notebook from the project root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.matrix_13f import (
    CUSIP_POLICY,
    MATRIX_CONTRACT_VERSION,
    MatrixParameters,
    decision_complete_quarters,
    materialize_full_history,
    validate_materialized_history,
)

## 1. Frozen identity and timing policy

The SEC's [Form 13F instructions](https://www.sec.gov/pdf/form13f.pdf) require the nine-character CUSIP in the information table, and the [Official List of Section 13(f) Securities](https://www.sec.gov/rules-regulations/staff-guidance/official-list-section-13f-securities) also publishes a nine-character field. The baseline therefore accepts a normalized, source-reported nine-character CUSIP only. Shorter or otherwise nonstandard values remain auditable but are not padded, checksum-completed, or silently merged.

The security key remains typed as `instrument_class:CUSIP`. This notebook materializes only the `cash_share_candidate` representation baseline; options and other channels remain separate inputs for later models. The 75 percent retained-weight cap and both 20-by-20 degree constraints are reapplied to a joint fixed point.

In [ ]:
PRODUCT_ROOT = PROJECT_ROOT / "data" / "full_history" / "13f" / "v1"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "model_inputs" / "13f_cash_share_pit" / "v1"
PARAMETERS = MatrixParameters(
    maximum_position_weight=0.75,
    minimum_assets_per_manager=20,
    minimum_managers_per_asset=20,
)

display(
    pd.DataFrame(
        {
            "setting": [
                "matrix contract",
                "CUSIP policy",
                "position cap",
                "minimum assets per manager",
                "minimum managers per asset",
            ],
            "value": [
                MATRIX_CONTRACT_VERSION,
                CUSIP_POLICY,
                PARAMETERS.maximum_position_weight,
                PARAMETERS.minimum_assets_per_manager,
                PARAMETERS.minimum_managers_per_asset,
            ],
        }
    )
)

## 2. Derive the full decision-complete history

No start or end quarter is hard-coded. Coverage begins only after the source was observed by quarter-end and ends only where the normal filing deadline is inside the local data freeze.

In [ ]:
eligible_quarters_local = decision_complete_quarters(PRODUCT_ROOT)
coverage_summary = pd.DataFrame(
    {
        "decision_complete_quarters": [len(eligible_quarters_local)],
        "first_report_date": [eligible_quarters_local["report_date_utc"].min()],
        "last_report_date": [eligible_quarters_local["report_date_utc"].max()],
    }
)
display(coverage_summary)

## 3. Materialize or safely resume every quarter

Each partition records the input fingerprint, source-code fingerprint, frozen parameters, pair-file hash, and aggregate audit. A matching completed quarter is reused; a stale or changed artifact is rebuilt. Set `FORCE_REBUILD` only when an intentional full recomputation is required.

In [ ]:
FORCE_REBUILD = False
run_summary_local = materialize_full_history(
    PRODUCT_ROOT,
    OUTPUT_ROOT,
    project_root=PROJECT_ROOT,
    parameters=PARAMETERS,
    force=FORCE_REBUILD,
)
display(
    pd.DataFrame(
        {
            "quarters": [len(run_summary_local)],
            "quarters_reused": [int(run_summary_local["cache_hit"].sum())],
            "total_sparse_pairs": [int(run_summary_local["pair_rows"].sum())],
        }
    )
)

## 4. Final full-history gates

Validation reads manifests, hashes, Parquet metadata, and saved aggregate checks without loading every pair table into memory. Modelling is blocked unless every violation count is zero.

In [ ]:
validation = validate_materialized_history(
    PRODUCT_ROOT,
    OUTPUT_ROOT,
    project_root=PROJECT_ROOT,
    parameters=PARAMETERS,
)
assert validation.checks["violations"].eq(0).all(), validation.checks
display(validation.checks)
display(
    validation.quarter_status[["report_quarter", "pair_rows", "managers", "assets"]]
)

## 5. Private Data Wrangler access

`matrix_dataset_local` represents the full partitioned history without reading it all into RAM. `matrix_pairs_local` loads one selected quarter as a DataFrame for Data Wrangler. Change `SELECTED_REPORT_QUARTER` to inspect another quarter. Do not display raw rows in a notebook that will be committed.

In [ ]:
pair_paths_local = sorted(OUTPUT_ROOT.glob("report_quarter=*/pairs.parquet"))
matrix_dataset_local = ds.dataset(
    [str(path) for path in pair_paths_local],
    format="parquet",
    partitioning="hive",
    partition_base_dir=str(OUTPUT_ROOT),
)
SELECTED_REPORT_QUARTER = str(
    eligible_quarters_local["report_date_utc"].max().to_period("Q")
)
selected_pair_path = (
    OUTPUT_ROOT / f"report_quarter={SELECTED_REPORT_QUARTER}" / "pairs.parquet"
)
matrix_pairs_local = pd.read_parquet(selected_pair_path)
display(
    pd.DataFrame(
        {
            "selected_report_quarter": [SELECTED_REPORT_QUARTER],
            "rows": [len(matrix_pairs_local)],
            "columns": [matrix_pairs_local.shape[1]],
        }
    )
)

## 6. Handoff to representation modelling

Once every gate passes, these local pair tables are the frozen ownership-state input for the first representation baseline. The next notebook should fit a simple binary-ownership factor model and evaluate representation quality separately from any return or trading backtest. A point-in-time market security master is still required before the universe can be called verified common equity or joined to returns.